### Приём 1. Чёткая системная инструкция

Начните с самого простого — **укажите модели, что она НЕ имеет права выдумывать:**

In [3]:
system_prompt = """
Отвечай ТОЛЬКО на основе предоставленного контекста.
Если в контексте нет информации — ответь: "Не знаю".
Не придумывай:
- task_key (он должен быть в формате PROJ-123),
- значения, которых нет в документации.
"""

context = ["Чанк1", "Чанк2"]
question = "Найди ответ в чанке1"

Затем передайте это в промпт:

In [4]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Контекст: {context}\nВопрос: {question}"}
]

! Это снижает галлюцинации на **60–80% даже у небольших моделей** (Mistral, Llama-3).

Но это **не гарантирует валидность JSON** — модель может «постараться» и вернуть {"task_key": "123"}.

### Приём 2. Генерация строго структурированного JSON

Чтобы LLM не «отклонялась от формата», **чётко** опишите ожидаемую структуру:

In [5]:
prompt = f"""
Сгенерируй задачу в Jira в формате JSON.
Обязательные поля:
- "task_key": строка в формате "PROJ-123" (пример: "BOT-42"),
- "summary": краткое описание (не более 60 символов).

Контекст: {context}
Вопрос: {question}
Ответ должен содержать ТОЛЬКО валидный JSON без пояснений.
"""

Такой подход превращает LLM из «чат-бота» в генератор структурированных данных, готовых к парсингу.

### Приём 3. Валидация

**Никогда не парсите JSON вручную.** Используйте Pydantic — он проверит **всё**, что вы ожидаете:

In [6]:
from pydantic import BaseModel, field_validator


class JiraTask(BaseModel):
    task_key: str
    summary: str

    @field_validator('task_key')
    def validate_task_key(cls, v):
        if not v or '-' not in v:
            raise ValueError('task_key must be in format PROJ-123')
        return v

    @field_validator('summary')
    def validate_summary(cls, v):
        if len(v) > 60:
            raise ValueError('summary must be no longer than 60 characters')
        if not v.strip():
            raise ValueError('summary cannot be empty')
        return v

Теперь валидация:

#### Так вы гарантированно отсечёте:

* пустые summary,
* неправильные task_key,
* слишком длинные описания,
* любые галлюцинации, не соответствующие схеме.

### Где и как использовать health-check?

В **продакшене** важно знать, что LLM всё ещё может генерировать валидный JSON.

Добавьте **health-check эндпоинт** в ваш сервис:

Этот эндпоинт подключается к Kubernetes, Docker Compose или мониторингу (Prometheus), чтобы автоматически перезапускать сервис при сбое.

In [23]:
from pydantic import BaseModel, field_validator

        
class Ticket(BaseModel):
    model_config = {"extra": "forbid"}
    id: str
    priority: str
    description: str

    @field_validator("id")
    def validate_id(cls, v: str) -> str:
        if not v.startswith('SUP-') or not len(v.split('-')[1]) == 4:
            raise ValueError('id must be in format SUP-1234')
        return v   
 
    @field_validator("priority")
    def validate_priority(cls, v: str) -> str:
        if v not in ["low", "medium", "high"]:
            raise ValueError('priority must be only in "low", "medium", "high"')
        return v  
    
    @field_validator("description")
    def validate_description(cls, v: str) -> str:
        if not 1 <= len(v) <= 100:
            raise ValueError('description must be have 1-100 symbols')
        return v 

class SupportRequest(BaseModel):
    model_config = {"extra": "forbid"}
    ticket: Ticket
    assignee: str
        
    @field_validator("assignee")
    def validate_assignee(cls, v: str) -> str:
        if v not in ["backend-team", "frontend-team", "security-team"]:
            raise ValueError
        return v

In [24]:
import json
from pydantic import ValidationError

# Пример ответа от модели (может приходить как строка или уже как dict)
llm_response = '{"ticket": {"id": "SUP-1234", "priority": "high", "description": "Ошибка входа"}, "assignee": "backend-team"}'

try:
    # 1. Если ответ в строке → парсим в dict. 
    # Если уже dict (например, response.json()), этот шаг можно пропустить.
    data = json.loads(llm_response) if isinstance(llm_response, str) else llm_response
    
    # 2. Валидация через вашу модель
    request = SupportRequest.model_validate(data)
    
    print("✅ Валидация прошла успешно!")
    # Доступ к данным:
    print(f"ID: {request.ticket.id} | Приоритет: {request.ticket.priority}")
    print(f"Команда: {request.assignee}")

except ValidationError as e:
    print("❌ Данные не соответствуют схеме:")
    for error in e.errors():
        # error['loc'] -> кортеж пути к полю, например ('ticket', 'id')
        print(f"  {'.'.join(str(l) for l in error['loc'])}: {error['msg']}")

except json.JSONDecodeError:
    print("❌ Ответ не является валидным JSON")

✅ Валидация прошла успешно!
ID: SUP-1234 | Приоритет: high
Команда: backend-team
